# 01 · Input latency — keystroke → PTY → render

The chain a keypress travels, per CLI kind, with the freeze signature called out
separately from the healthy-but-quiet case.

**What this measures.** `input/keystroke` (GUI, xterm.js `onData`) → `input/pty`
(daemon) → `input/render` (GUI, bytes about to be painted). The three legs are
emitted by *two different processes*, so they share only `session_path` and a
timestamp.

⛔ **`input/pty` without `input/keystroke` is not a defect.** Agent writes arrive
through app-control and never touch xterm.js, so on a host where agents are
working and nobody is typing, the pty leg fires alone. This notebook reports
INSUFFICIENT DATA rather than inventing a latency from it. See
`docs/observability.md` §4.1.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) != "notebooks" else os.getcwd()))
sys.path.insert(0, os.path.abspath("."))
import ytrace_helpers as H

WINDOW = os.environ.get("YGG_NOTEBOOK_WINDOW", "30m")
HOST = H.GUI_HOST
H.describe_source(HOST, WINDOW)

In [ ]:
# Raw records — `tail` is the only verb that exposes per-record payloads.
records = H.tail(HOST, since=WINDOW, category="input")
by_probe = {}
for r in records:
    by_probe.setdefault(H.probe_key(r), []).append(r)

print(f"{len(records)} input records over {WINDOW}")
for k in sorted(by_probe):
    print(f"  {k:20} {len(by_probe[k]):6d}")
for leg in ("input/keystroke", "input/pty", "input/render"):
    if leg not in by_probe:
        print(f"  {leg:20} {'0':>6}  (leg silent in this window)")

In [ ]:
# Session kind is derived from the path scheme, never from a second field.
def session_kind(path):
    if not path:
        return "unknown"
    scheme = str(path).split("://", 1)[0]
    return scheme or "unknown"

def sess(r):
    p = r.get("payload") or {}
    return p.get("session_path") if isinstance(p, dict) else None

keystrokes = by_probe.get("input/keystroke", [])
ptys       = by_probe.get("input/pty", [])
renders    = by_probe.get("input/render", [])

kinds = H.Counter(session_kind(sess(r)) for r in keystrokes)
print("keystrokes by session kind:", dict(kinds) or "(none)")

In [ ]:
# Pair each keystroke with the next pty/render on the SAME session_path.
# A bounded window keeps an unrelated later event from being credited as the
# answer to this keystroke.
PAIR_WINDOW_MS = 5_000

def index_by_session(rows):
    out = {}
    for r in rows:
        out.setdefault(sess(r), []).append(r)
    for v in out.values():
        v.sort(key=lambda x: x.get("ts_ms", 0))
    return out

pty_idx, render_idx = index_by_session(ptys), index_by_session(renders)

def next_after(idx, path, ts):
    for r in idx.get(path, []):
        gap = r.get("ts_ms", 0) - ts
        if 0 <= gap <= PAIR_WINDOW_MS:
            return gap
    return None

legs = {"keystroke_to_pty": [], "pty_to_render": [], "keystroke_to_render": []}
orphans = []
per_kind = {}
for k in keystrokes:
    path, ts = sess(k), k.get("ts_ms", 0)
    to_pty = next_after(pty_idx, path, ts)
    to_render = next_after(render_idx, path, ts)
    if to_pty is None:
        orphans.append(k)
    else:
        legs["keystroke_to_pty"].append(to_pty)
        per_kind.setdefault(session_kind(path), []).append(to_pty)
    if to_render is not None:
        legs["keystroke_to_render"].append(to_render)
        if to_pty is not None:
            legs["pty_to_render"].append(to_render - to_pty)

for name, vals in legs.items():
    print(f"{name:22}", H.percentiles(vals))
print(f"\nkeystrokes with NO pty within {PAIR_WINDOW_MS} ms: {len(orphans)} of {len(keystrokes)}")

In [ ]:
# Per-CLI-kind breakdown — the question is whether one session kind is slower.
rows = []
for kind, vals in sorted(per_kind.items()):
    st = H.percentiles(vals)
    rows.append({"session_kind": kind, "n": st.get("n", 0), "p50_ms": st.get("p50"),
                 "p95_ms": st.get("p95"), "p99_ms": st.get("p99"), "max_ms": st.get("max")})
print(H.table(rows, ["session_kind", "n", "p50_ms", "p95_ms", "p99_ms", "max_ms"]) if rows
      else "(no paired keystrokes — nothing to break down)")

In [ ]:
RENDER_LAG_WARN_MS = 50.0   # a keypress the eye notices
RENDER_LAG_FAIL_MS = 150.0
ORPHAN_WARN_RATIO  = 0.05   # 1 in 20 keystrokes losing its pty leg

v = H.Verdict("Input latency: keystroke -> PTY -> render")

if not keystrokes:
    v.note(H.UNKNOWN, "human input observed",
           "no `input/keystroke` in the window. The pty leg alone is the ordinary "
           "shape of agent writes through app-control; it is NOT a latency and NOT a "
           "freeze. Re-run while someone is typing into a terminal view.")
    v.note(H.UNKNOWN, "input/pty seen alone", f"{len(ptys)} pty records with no keystroke to pair")
else:
    k2p = H.percentiles(legs["keystroke_to_pty"])
    k2r = H.percentiles(legs["keystroke_to_render"])
    v.check("keystroke -> pty p95", k2p.get("p95"), f"<= {RENDER_LAG_WARN_MS} ms",
            warn_over=RENDER_LAG_WARN_MS, fail_over=RENDER_LAG_FAIL_MS, n=k2p.get("n", 0))
    v.check("keystroke -> render p95", k2r.get("p95"), f"<= {RENDER_LAG_FAIL_MS} ms",
            warn_over=RENDER_LAG_WARN_MS, fail_over=RENDER_LAG_FAIL_MS, n=k2r.get("n", 0))
    ratio = len(orphans) / len(keystrokes)
    v.check("keystrokes losing the pty leg", ratio, f"<= {ORPHAN_WARN_RATIO:.0%}",
            warn_over=ORPHAN_WARN_RATIO, fail_over=0.20,
            detail="the input-freeze signature: typed but never reached the daemon",
            n=len(keystrokes))

if not renders:
    v.note(H.UNKNOWN, "input/render leg", "silent in this window — the render leg cannot be timed")
v.show()